In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Load dataset
df = pd.read_csv("runtime_dataset_final.csv")

# Drop missing values
df = df.dropna()

# Features and target
X = df.drop("runtime_sec", axis=1)
y = df["runtime_sec"]

# Encode categorical columns
for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pipeline (Scaling + Model)
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVR())
])

# Hyperparameter grid
param_grid = {
    "svm__kernel": ["rbf", "linear", "poly"],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", "auto"],
    "svm__epsilon": [0.01, 0.1, 0.5]
}

# Grid search
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# Train model
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

# Best model
best_model = grid.best_estimator_

# Predict
y_pred = best_model.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

Best Parameters: {'svm__C': 100, 'svm__epsilon': 0.5, 'svm__gamma': 'auto', 'svm__kernel': 'rbf'}
RMSE: 9.066176936227906
R2 Score: 0.9763016183367386
